In [38]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
class Value:
    # Value类，打算实现图的自动建立，标签和数据的运算的运算方法，还有梯度的反向传播的自动计算，这些目前都已实现
    #运算方法如下，还有一些未实现的
    #已实现的有： 加法（包括交换律可交换，并且可处理浮点类型的类 ），乘法（和加法一样，可处理浮点类型），tanh函数。
    #想要实现的还有：除法，幂运算，减法，和指数函数比如e的x次方（✅），log函数，三角函数（这个可以用指数函数做到）
    
    #初始化的一些参数，data是数据，label是标签，_children是子节点，_op是运算符，grad是梯度，_backward是反向传播的函数，注意初始化的时候设置的是一个空函数
    def __init__(self,data,label='',_children=(),_op=''):
        self.data=data
        self._prev=set(_children)
        self.label=label
        self.grad=0.0
        self._op=_op
        self._backward=lambda:None
    #打印函数，规定print(value)时的输出格式
    def __repr__(self):
        return f"Value(data={self.data}, label={self.label})"
    
    #加法函数，规定了正向的value对象加value对象的加法运算，同时定义了反向传播的梯度计算
    def __add__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data+other.data,_children=(self,other),_op='+')
        def _backward():
            self.grad+=out.grad*1.0
            other.grad+=out.grad*1.0
        out._backward=_backward
        return out
    #反向加法，当value对象在加法运算中处于右边时，调用该函数
    def __radd__(self,other):
            return self+other

    #乘法函数，实现乘法及其相应的反向传播
    def __mul__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data*other.data,_children=(self,other),_op='*')
        def _backward():
            self.grad+=out.grad*other.data
            other.grad+=out.grad*self.data
        out._backward=_backward
        return out
    #反向乘法
    def __rmul__(self,other):
        return self*other

    #除法,目前完成状态❌，仔细想了想除法是乘以倒数，所以可以当成乘法，但是倒数涉及到x的幂运算，所以要先实现幂运算
    def __truediv__(self,other):
        other=other if isinstance(other, Value) else Value(other)
        out=Value(self.data)

    #接下来开始实现指数函数，e的x次方，定义了正向计算和反向传播
    def exp(self):
        x=self.data
        out=Value(math.exp(x),_children=(self,),_op='exp')
        def _backward():
            self.grad+=out.grad*math.exp(x)
        out._backward=_backward
        return out
    #成功实现

    #幂运算
    def __pow__(self,other):
        assert isinstance(other,(int,float)), "only supporting int/float powers for now"
        out=Value(self.data**other,_children=(self,),_op=f'**{other}')
        def _backward():
            self.grad+=out.grad*other*(self.data**(other-1))
        out._backward=_backward
        return out

    #双曲tanh函数，定义了tanh函数的正向计算和反向传播
    def tanh(self):
        x=self.data
        t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
        out=Value(t,_children=(self,),_op='tanh')
        def _backward():
            self.grad+=out.grad*(1-t**2)
        out._backward=_backward
        return out

    #反向传播函数，包括了图的DFS算法，拓扑排序，这样就可以保证传播的时候梯度顺序不搞错
    def backward(self):
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad=1.0
        for node in reversed(topo):
            node._backward()


In [40]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})  # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name=uid, label="{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


In [41]:
# a=Value(3.0,'a')
# b=Value(-2.0,'b')
# c=a+b;c.label='c'

#来个复杂点的

#iputs of x
x1=Value(2.0,label='x1')
x2=Value(4.0,label='x2')
x3=Value(-3.0,label='x3')

#inputs of whights
w1=Value(3.0,label='w1')
w2=Value(-3.0,label='w2')
w3=Value(0.0,label='w3')

#bias of the neuron

b=Value(6.8813735870195432,label='b')
#计算流程
x1w1=x1*w1;x1w1.label='x1w1'
x2w2=x2*w2;x2w2.label='x2w2'
x1w1x2w2=x1w1+x2w2;x1w1x2w2.label='x1w1+x2w2'
x3w3=x3*w3;x3w3.label='x3w3'
x1w1x2w2x3w3=x1w1x2w2+x3w3;x1w1x2w2x3w3.label='x1w1+x2w2+x3w3'

h=x1w1x2w2x3w3+b;h.label='h'
m=h.tanh();m.label='m'



In [42]:
# def tanh(x):
   
#     t=(math.exp(2*x)-1)/(math.exp(2*x)+1)
#     return t
# jj=tanh(0.6)   
# print(jj)

In [ ]:
#接下来是反向传递的示例，看看能不能正常运行backward，这个backward是之前写的空函数，但是我们其实是手动算的导数相当于，因为这里的每一个操作都是数学那里可以写的出来导数的操作，所以我们可以把导数记录下来


# m.grad=1.0
# m.backward()
# h.backward()
# b.backward()
# x1w1x2w2x3w3.backward()
# x1w1x2w2.backward()
# x2w2.backward()
# x1w1.backward()
# x3w3.backward()

# w3.backward()
# x3.backward()

# x1.backward()
# w1.backward()
# x2.backward()
# w2.backward()  

#已经封装好了，这些可以注释掉了可以全部换成快速的方法，必要的时候可以取消注释来验算
m.backward()

draw_dot(m)



In [44]:
#接下来是拓扑排序实现自动反向传播,这里待会可以封装到value类里,
# topo=[]
# visited=set()
# def build_topo(v):
#     if v not in visited:
#         visited.add(v)
#         for child in v._prev:
#             build_topo(child)
#         topo.append(v)
# for node in reversed(topo):
#     node.backward()
#写完封装，然后把这一段注释掉